In [1]:
!pip install  torchao==0.10.0 transformers datasets


In [2]:
import torch

import data
import llm
import model
import spsa
import json



#################################
## CONSTANTS
#################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

CONFIG_BASIC_RUN = {
        "llm": "albert",
        "dataset": "imdb",
}


#################################
## Class definitions
#################################

  
#################################
## Private Functions
#################################

#################################
## Public Functions
#################################

def train(model, dataloader, loss_fn, epoch):
    total_loss = 0.0
    for batch in dataloader:
        attention_mask = batch["attention_mask"].to(device) if "attention_mask" in batch else None
        loss = model.step(batch["input_ids"].to(device), batch["labels"].to(device), attention_mask=attention_mask, epoch=epoch)
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

Device:  cuda


In [ ]:
llm_model = llm.get_model("albert")
loss_fn = torch.nn.CrossEntropyLoss()
layers = torch.nn.Sequential(
    torch.nn.Linear(768, 256),
    torch.nn.GELU(),
    torch.nn.Linear(256, 128),
    torch.nn.GELU(),
    torch.nn.Linear(128, 2)
)
model = model.Model(llm_model["model"].to(device), loss_fn, layers).to(device)

dataset = data.get_dataset("imdb", llm_model["tokenizer"], batch_size=32)

model= spsa.SPSA(model, lr=0.05, delta=0.01, loss_fn=loss_fn,num_estimates=1)
model.init_sgd(lr=0.1)

log = []
for i in range(1000):
    log.append(train(model, dataset, loss_fn,i+1))
    print(f"Epoch {i}, Test loss {log[-1]}")

path = "results/spsa.json"
with open(path, "r") as f:
    json.dump(log, f, indent=1)

2026-05-04 08:31:14.892653: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777883474.906885  302426 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777883474.911508  302426 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-04 08:31:14.926613: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of the model checkpoint at albert/albert-base-v2 were not used when initializing AlbertForMask

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]


/home/johaanto/work/SPSA-pytorch/data.py:48: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch 0, Test loss 0.008900238016203555
